In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import sys
sys.path.append("../")
import pandas as pd
from utils import ca as cu, behave as bu, plot as pu, db as db
import numpy as np
from paths.config import M2PConfig
import matplotlib.pyplot as plt
import scipy
import plotly.express as px
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

In [3]:
cfg = M2PConfig()

# Load good data only
df_exps, df_roi, df_ca = db.get_roi_ca_data(cfg)

Excluded 26221 bad 2p frames 0.54%


In [4]:
grp_df_cell = df_ca.groupby(['exp_id', 'animal_id', 'roi_id', 'celltype', 'roi_type'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf

def permutation_acf_threshold(data, n_lags, n_permutations=200, percentile=95):
    """
    Calculate significance threshold for ACF using random circular shifts.
    
    Parameters:
    -----------
    data : array-like
        Time series data
    n_lags : int
        Number of lags to calculate
    n_permutations : int
        Number of random rolls/shifts
    percentile : float
        Percentile for threshold (default 95)
    
    Returns:
    --------
    acf_values : array
        ACF values for each lag
    threshold : float
        Single significance threshold for all lags
    """
    min_shift = 300
    n = len(data) - min_shift
    acf_values = acf(data, nlags=n_lags)
    
    # Store all ACF values from permutations
    all_acf_values = []
    
    for i in range(n_permutations):
        # Randomly roll/shift the time series
        shift = np.random.randint(min_shift, n)  # Random shift (avoid 0)
        rolled_data = np.roll(data, shift)
        
        # Calculate ACF for rolled data
        rolled_acf = acf(rolled_data, nlags=n_lags)
        
        # Store all ACF values (excluding lag 0 which is always 1)
        all_acf_values.extend(np.abs(rolled_acf[1:]))
    
    # Calculate the percentile threshold
    threshold = np.percentile(all_acf_values, percentile)
    
    return acf_values, threshold


# Storage for averaged ACF plots
acf_penk = []
acf_non_penk = []
threshold_penk = []
threshold_non_penk = []

n_lags = 100

# YOUR MODIFIED CODE:
for (exp_id, animal_id, roi_id, cell_type, roi_type), df_group in grp_df_cell:

    print((exp_id, animal_id, roi_id, cell_type, roi_type))

    ca = df_group[cu.CA_DECONV_NORM]

    # Calculate ACF with permutation threshold
    acf_values, threshold = permutation_acf_threshold(ca.values, 
                                                       n_lags=n_lags, 
                                                       n_permutations=200, 
                                                       percentile=95)

    # Store ACF values based on cell type
    if cell_type == 'penk':
        acf_penk.append(acf_values)
        threshold_penk.append(threshold)
    else:
        acf_non_penk.append(acf_values)
        threshold_non_penk.append(threshold)

    # Plot the auto-correlation
    fig, axes = plt.subplots(2, 1, figsize=(10, 8))
    
    # Plot the original time series
    axes[0].plot(ca, linewidth=1.5)
    axes[0].set_title('Time Series Data (ca)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Value')
    axes[0].grid(True, alpha=0.3)

    # Plot autocorrelation function with permutation threshold
    lags = np.arange(len(acf_values))
    axes[1].vlines(lags, [0], acf_values, colors='steelblue', linewidth=1.5)
    axes[1].axhline(y=0, color='black', linewidth=0.8)
    
    # Plot threshold lines (positive and negative)
    axes[1].axhline(y=threshold, color='red', linewidth=1.5, linestyle='--', 
                    label=f'95% threshold = {threshold:.4f}')
    axes[1].axhline(y=-threshold, color='red', linewidth=1.5, linestyle='--')
    
    # Shade the significance region
    axes[1].fill_between(lags, -threshold, threshold, alpha=0.2, color='gray', 
                          label='Non-significant region')
    
    axes[1].set_title('Autocorrelation Function (ACF) - Permutation Test', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Lag')
    axes[1].set_ylabel('Autocorrelation')
    axes[1].set_xlim(-1, n_lags + 1)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


# PLOT AVERAGED ACF AT THE END
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot averaged ACF for 'penk' cells
if acf_penk:
    acf_penk_mean = np.mean(acf_penk, axis=0)
    acf_penk_sem = np.std(acf_penk, axis=0) / np.sqrt(len(acf_penk))
    threshold_penk_mean = np.mean(threshold_penk)
    
    lags = np.arange(len(acf_penk_mean))
    axes[0].vlines(lags, [0], acf_penk_mean, colors='steelblue', linewidth=2)
    axes[0].axhline(y=0, color='black', linewidth=0.8)
    
    # Add SEM as error bars (optional)
    axes[0].fill_between(lags, acf_penk_mean - acf_penk_sem, acf_penk_mean + acf_penk_sem, 
                         alpha=0.3, color='steelblue', label='SEM')
    
    # Plot threshold
    axes[0].axhline(y=threshold_penk_mean, color='red', linewidth=1.5, linestyle='--', 
                    label=f'95% threshold = {threshold_penk_mean:.4f}')
    axes[0].axhline(y=-threshold_penk_mean, color='red', linewidth=1.5, linestyle='--')
    axes[0].fill_between(lags, -threshold_penk_mean, threshold_penk_mean, alpha=0.2, color='gray')
    
    axes[0].set_title(f'Average ACF - PENK cells (n={len(acf_penk)})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Lag')
    axes[0].set_ylabel('Autocorrelation')
    axes[0].set_xlim(-1, n_lags + 1)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

# Plot averaged ACF for non-'penk' cells
if acf_non_penk:
    acf_non_penk_mean = np.mean(acf_non_penk, axis=0)
    acf_non_penk_sem = np.std(acf_non_penk, axis=0) / np.sqrt(len(acf_non_penk))
    threshold_non_penk_mean = np.mean(threshold_non_penk)
    
    lags = np.arange(len(acf_non_penk_mean))
    axes[1].vlines(lags, [0], acf_non_penk_mean, colors='orange', linewidth=2)
    axes[1].axhline(y=0, color='black', linewidth=0.8)
    
    # Add SEM as error bars (optional)
    axes[1].fill_between(lags, acf_non_penk_mean - acf_non_penk_sem, acf_non_penk_mean + acf_non_penk_sem, 
                         alpha=0.3, color='orange', label='SEM')
    
    # Plot threshold
    axes[1].axhline(y=threshold_non_penk_mean, color='red', linewidth=1.5, linestyle='--', 
                    label=f'95% threshold = {threshold_non_penk_mean:.4f}')
    axes[1].axhline(y=-threshold_non_penk_mean, color='red', linewidth=1.5, linestyle='--')
    axes[1].fill_between(lags, -threshold_non_penk_mean, threshold_non_penk_mean, alpha=0.2, color='gray')
    
    axes[1].set_title(f'Average ACF - Non-PENK cells (n={len(acf_non_penk)})', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Lag')
    axes[1].set_ylabel('Autocorrelation')
    axes[1].set_xlim(-1, n_lags + 1)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"PENK cells: {len(acf_penk)} cells averaged")
print(f"Non-PENK cells: {len(acf_non_penk)} cells averaged")